### Qwen3-1.7B Egyptian Arabic Customer Service Fine-Tuning with LoRA


This project fine-tunes the Qwen/Qwen3-1.7B large language model using Low-Rank Adaptation (LoRA) on a custom Egyptian Arabic customer service dataset, enabling the model to provide natural, context-aware, and helpful responses to customer inquiries.

## Setup the model
### Libraries

In [24]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
import json
import torch
from peft import get_peft_model, LoraConfig, TaskType
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from peft import PeftModel
from huggingface_hub import HfApi


## Step 1: Load and Preprocess the Dataset

In [41]:
import json

with open("CustomerData.json", "r", encoding="utf-8") as f:
    data = json.load(f)

print(data[0])

{'conversations': [{'role': 'user', 'content': 'السلام عليكم، عندي مشكلة في الأوردر.'}, {'role': 'assistant', 'content': 'وعليكم السلام، تحت أمرك. ممكن تقولي رقم الأوردر أو توضحلي المشكلة حصلت إزاي؟'}]}


In [7]:
dataset = Dataset.from_list(data)


In [8]:
model_name = "Qwen/Qwen3-1.7B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [9]:
def chat_format(example):
  return{
      "text": tokenizer.apply_chat_template(example["conversations"], tokenize = False, add_generation_prompt = False)
  }


In [10]:
def tokenize(example):
  result = tokenizer(example["text"], truncation = True, padding = "max_length", max_length= 512)
  result["labels"] = result["input_ids"].copy()  # Labels are the same as input_ids for causal LM
  return result


In [11]:
dataset = dataset.map(chat_format, remove_columns="conversations")
dataset = dataset.map(tokenize, remove_columns="text")
dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 257
})

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype= torch.float16,
    device_map="auto"
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


In [13]:
dataset

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 257
})

##  Step 2: LoRA Adapter Configuration

In [15]:
lora_config = LoraConfig(
    r=8,
    lora_alpha= 16,
    lora_dropout= 0.05,
    bias= "none",
    task_type= TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)



##  Step 3: Training

In [16]:
training_args= TrainingArguments(
    output_dir= "qwen-Customer-Service-lora",
    per_device_train_batch_size = 2,
    gradient_accumulation_steps=2,              # Accumulate gradients over 2 steps
    num_train_epochs=5,                         # Train for 5 full epochs
    learning_rate=2e-4,                         # Learning rate
    fp16=True,                                  # Use mixed precision training
    logging_steps=10,                           # Log training info every 10 steps
    save_steps=50,                              # Save checkpoint every 50 steps
    save_total_limit=2,                         # Keep only the last 2 checkpoints
    report_to="none",                           # Disable external logging (e.g., wandb)
    remove_unused_columns=False                 # Keep all columns (important for LoRA)
)
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=dataset
)

trainer.train()



TrainOutput(global_step=325, training_loss=1.8506723756056565, metrics={'train_runtime': 374.2548, 'train_samples_per_second': 3.433, 'train_steps_per_second': 0.868, 'total_flos': 5570012617113600.0, 'train_loss': 1.8506723756056565, 'epoch': 5.0})

## Step 4: Save and Reload Model

In [17]:
model.save_pretrained("./qwen-Customer-Service-lora")
tokenizer.save_pretrained("./qwen-Customer-Service-lora")

('./qwen-Customer-Service-lora/tokenizer_config.json',
 './qwen-Customer-Service-lora/chat_template.jinja',
 './qwen-Customer-Service-lora/tokenizer.json')

## Step 5: Inference

In [18]:
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", torch_dtype=torch.float16)
model = PeftModel.from_pretrained(model, "./qwen-Customer-Service-lora")
model.eval()

tokenizer = AutoTokenizer.from_pretrained("./qwen-Customer-Service-lora")


In [23]:
prompt =  "الأوردر لسه موصلش."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=512)


print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
الأوردر لسه موصلش.
assistant
<think>

</think>

آسفين على التأخير. ابعتلي رقم الأوردر وأنا أراجع مع الفريق للتأكد من حالة الأوردر.


## Step 6: Upload to Hugging Face Hub


In [20]:
from huggingface_hub import login

login()


In [25]:
repo_id = "Eldenary/qwen-Customer-Service-lora"

api = HfApi()

# create a repo if not exists.
api.create_repo(

    repo_id=repo_id,
    repo_type="model",
    exist_ok=True
)

# upload files
api.upload_folder(
    folder_path="./qwen-Customer-Service-lora",
    repo_id=repo_id,
    repo_type="model"
)

CommitInfo(commit_url='https://huggingface.co/Eldenary/qwen-Customer-Service-lora/commit/7827e74ed2c89e376d8a3f475937c939c5c56444', commit_message='Upload folder using huggingface_hub', commit_description='', oid='7827e74ed2c89e376d8a3f475937c939c5c56444', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Eldenary/qwen-Customer-Service-lora', endpoint='https://huggingface.co', repo_type='model', repo_id='Eldenary/qwen-Customer-Service-lora'), pr_revision=None, pr_num=None)

## Step 7: Use from Anywhere

In [ ]:
base_model = "Qwen/Qwen3-1.7B"
adapter_id = "Eldenary/qwen-Customer-Service-lora"

tokenizer = AutoTokenizer.from_pretrained(base_model)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    torch_dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(model, adapter_id)
model.eval()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 2048)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Linear(in_fea

In [40]:
prompt =  "عايز أغير عنوان الشحن."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
  outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample= True,
        top_p=0.95,
        temperature=0.4
        )



print(tokenizer.decode(outputs[0], skip_special_tokens=True))

user
عايز أغير عنوان الشحن.
assistant
<think>

</think>

لو الطلب موضح فيه العنوان الحالي، ابعتلي رقم الأوردر وأنا أراجع إمكانية تغيير العنوان.
